In [3]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset --------
class RelativeSpeedDataset200D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11)

                try:
                    feat = np.concatenate([
                        d, o, own_acc, d1, d2,
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20]
                    ])
                except:
                    continue

                if feat.shape[0] != 200:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid

# -------- 改良版 LSTMモデル（未来なし・正規化なし） --------
class NonNormLSTMModel(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=256, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), 20, 10)
        out, _ = self.lstm(x)
        last = out[:, -1, :]         # 未来は見ない
        dropped = self.dropout(last)
        return self.fc(dropped).squeeze(1)

# -------- 学習ループ --------
def train_nonnorm_lstm(dataset, save_path="model_nonnorm_lstm.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts), list(sids)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = NonNormLSTMModel().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.SmoothL1Loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, threshold=0.0001
    )

    best_val_loss = float('inf')
    patience = 40
    min_delta = 0.0005
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if best_val_loss - val_loss > min_delta:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            counter = 0
        else:
            counter += 1
            print(f"⏸ No significant improvement. Patience: {counter}/{patience}")
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行部 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset200D(
        annot_root="../train/train_annotations",
        distance_json_path="../distance3/distance_estimates_corrected.json",
        max_items=7500
    )

    model = train_nonnorm_lstm(dataset, save_path="model_nonnorm_lstm.pth")
    print("✅ 学習完了: model_nonnorm_lstm.pth に保存しました")


[Train 1]: 100%|██████████| 93/93 [00:00<00:00, 128.24it/s]


Epoch 1 | Train Loss: 3.8401 | Val Loss: 3.3837
✅ Saved model to model_nonnorm_lstm.pth (val_loss=3.3837)


[Train 2]: 100%|██████████| 93/93 [00:00<00:00, 133.22it/s]


Epoch 2 | Train Loss: 2.7939 | Val Loss: 0.4973
✅ Saved model to model_nonnorm_lstm.pth (val_loss=0.4973)


[Train 3]: 100%|██████████| 93/93 [00:00<00:00, 134.59it/s]


...

Epoch 98 | Train Loss: 0.1329 | Val Loss: 0.1726
⏸ No significant improvement. Patience: 4/40


[Train 99]: 100%|██████████| 93/93 [00:00<00:00, 138.59it/s]


Epoch 99 | Train Loss: 0.1325 | Val Loss: 0.1790
⏸ No significant improvement. Patience: 5/40


[Train 100]: 100%|██████████| 93/93 [00:00<00:00, 140.50it/s]


Epoch 100 | Train Loss: 0.1321 | Val Loss: 0.1874
⏸ No significant improvement. Patience: 6/40
✅ 学習完了: model_nonnorm_lstm.pth に保存しました
